# 6TH : Linear classification

In [ ]:
import pickle
import numpy as np
import os
import pandas as pd
from sklearn.metrics import confusion_matrix

In [2]:
def load_cifar10_data(file_path):
    with open(file_path, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data']
    labels = batch[b'labels']
    return images, np.array(labels)

data_dir = 'cifar-10-batches-py'

all_train_images = []
all_train_labels = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    images, labels = load_cifar10_data(batch_path)
    all_train_images.append(images)
    all_train_labels.append(labels)

all_train_images = np.concatenate(all_train_images, axis=0)
all_train_labels = np.concatenate(all_train_labels, axis=0)

test_path = os.path.join(data_dir, 'test_batch')
all_test_images, all_test_labels = load_cifar10_data(test_path)

print(f"Train dataset: {all_train_images.shape}")
print(f"Test dataset: {all_test_images.shape}")

Train dataset: (50000, 3072)
Test dataset: (10000, 3072)


In [ ]:
class LinearClassifier:
    def __init__(self, input_dim, num_classes):
        self.W = np.random.randn(num_classes, input_dim)
        self.b = np.zeros(num_classes)
        
    def dot(self, X):
        return (np.dot(self.W, X.T)).T + self.b
        
    def softmax(self, scores):
        shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
        exp_scores = np.exp(shifted_scores)
        probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return probs
        
    def predict(self, X):
        scores = self.dot(X)
        probs = self.softmax(scores)
        return np.argmax(probs, axis=1)

    def loss(self, X, y):
        scores = self.dot(X)
        N = X.shape[0]
        correct_scores = scores[np.arange(N), y]
        margins = np.maximum(0, scores - correct_scores[:, None] + 1)
        margins[np.arange(N), y] = 0

        loss = np.sum(margins) / N
        return loss

In [4]:
input_dim = all_train_images.shape[1]
num_classes = 10

lc = LinearClassifier(input_dim, num_classes)

preds = lc.predict(all_test_images)

accuracy = np.mean(preds == all_test_labels)

loss = lc.loss(all_test_images, all_test_labels)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Loss: {loss:.4f}")

Accuracy: 9.92%
Loss: 36314.8978


In [5]:
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
cm = confusion_matrix(all_test_labels, preds)

cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

cm_df

,airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
airplane,867,0,6,2,0,5,3,3,0,114
automobile,794,0,39,7,0,16,3,15,0,126
bird,898,0,7,3,0,9,5,10,0,68
cat,884,1,20,6,0,8,8,14,0,59
deer,933,1,7,0,1,5,0,11,1,41
dog,911,0,10,1,0,4,3,19,0,52
frog,900,0,38,3,0,7,2,11,0,39
horse,886,0,13,4,0,8,0,15,1,73
ship,786,0,11,2,0,13,0,3,0,185
truck,869,0,6,5,0,19,1,10,0,90
